# Ejemplo: Registro de Usuarios
**Simulador de Planta Móvil — Backend**

Este notebook demuestra el flujo completo de registro de usuarios usando el módulo `backend.auth.register`.  
Los pasos cubiertos son:

1. Importar librerías y módulos del proyecto
2. Definir datos de prueba (válidos e inválidos)
3. Llamar a la función de registro
4. Validar la respuesta
5. Manejar errores de validación
6. Probar registro duplicado

> **Nota:** Ejecutar desde la raíz del proyecto para que las importaciones del paquete `backend` funcionen correctamente.


---
## 1. Importar librerías y módulos


In [1]:
import sys
import os

# Agregar la raíz del proyecto al path para poder importar el paquete 'backend'
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from backend.auth.register import registrar_usuario
from backend.auth.models import UsuarioRegistro, UsuarioRespuesta

print("Módulos cargados correctamente ✓")


Módulos cargados correctamente ✓


---
## 2. Definir datos de prueba

Se preparan payloads con datos válidos e inválidos para cubrir todos los casos del registro.


In [2]:
# ── Datos válidos ──────────────────────────────────────────────
usuario_valido = UsuarioRegistro(
    nombre="Tilina",
    correo="sologym@correo.com",
    password="clave1234"
)

# ── Datos inválidos para distintos escenarios ───────────────────
nombre_vacio     = UsuarioRegistro(nombre="  ",         correo="otro@correo.com",   password="clave1234")
password_corta   = UsuarioRegistro(nombre="Carlos",     correo="carlos@correo.com", password="123")
correo_invalido  = UsuarioRegistro(nombre="Luis",       correo="no-es-un-correo",   password="clave1234")
usuario_valido_2 = UsuarioRegistro(nombre="Pedro",      correo="pedro@correo.com",  password="otraClaveSegura")

print("Datos de prueba definidos ✓")


Datos de prueba definidos ✓


In [3]:
# Limpiar el usuario de prueba antes de cada ejecución del notebook
# Solo borra el correo específico para no afectar a otros usuarios
from backend.database import get_connection, init_db

init_db()
conn = get_connection()
cursor = conn.cursor()
cursor.execute('DELETE FROM "user" WHERE email = %s', ("sologym@correo.com",))
cursor.execute('DELETE FROM "user" WHERE email = %s', ("pedro@correo.com",))
conn.commit()
conn.close()
print("Registros de prueba limpiados ✓")

Registros de prueba limpiados ✓


---
## 3. Llamar a la función de registro

Se invoca `registrar_usuario()` con datos válidos y se captura la respuesta.


In [4]:
resultado = registrar_usuario(usuario_valido)
resultado


{'exito': True,
 'mensaje': 'Usuario registrado exitosamente.',
 'usuario': UsuarioRespuesta(id=10, nombre='Tilina', correo='sologym@correo.com', online=False, rol_admin=False, creado='None')}

---
## 4. Validar la respuesta

Se verifica que el registro fue exitoso y que la respuesta contiene los campos esperados.


In [5]:
assert resultado["exito"] is True, "Se esperaba registro exitoso"
assert resultado["usuario"] is not None, "Debe retornar un objeto usuario"
assert resultado["usuario"].correo == "sologym@correo.com"

u = resultado["usuario"]
print(f"✓ Registro validado")
print(f"  id        : {u.id}")
print(f"  nombre    : {u.nombre}")
print(f"  correo    : {u.correo}")
print(f"  online    : {u.online}")
print(f"  rol_admin : {u.rol_admin}")
print(f"  creado    : {u.creado}")

✓ Registro validado
  id        : 10
  nombre    : Tilina
  correo    : sologym@correo.com
  online    : False
  rol_admin : False
  creado    : None


---
## 5. Manejar errores de validación

Se prueban los casos donde el input es inválido: nombre vacío, contraseña corta y correo con formato incorrecto.


In [6]:
casos_invalidos = [
    ("Nombre vacío",                  nombre_vacio),
    ("Contraseña muy corta",          password_corta),
    ("Correo con formato inválido",   correo_invalido),
]

for descripcion, datos in casos_invalidos:
    res = registrar_usuario(datos)
    assert res["exito"] is False, f"Se esperaba error en: {descripcion}"
    print(f"✗ {descripcion}")
    print(f"  → {res['mensaje']}\n")


✗ Nombre vacío
  → El nombre no puede estar vacío.

✗ Contraseña muy corta
  → La contraseña debe tener al menos 8 caracteres.

✗ Correo con formato inválido
  → El formato del correo no es válido.



---
## 6. Probar registro duplicado

Se intenta registrar exactamente el mismo correo dos veces.  
El segundo intento debe ser rechazado con el mensaje `"El correo ya está registrado."`.


In [7]:
# El correo de Tilina ya fue registrado en la sección 3 — el segundo intento debe fallar
duplicado = UsuarioRegistro(
    nombre="Tilina Copia",
    correo="sologym@correo.com",   # mismo correo
    password="otraClave99"
)

res_duplicado = registrar_usuario(duplicado)

assert res_duplicado["exito"] is False
assert "ya está registrado" in res_duplicado["mensaje"]

print(f"✗ Registro duplicado rechazado correctamente")
print(f"  → {res_duplicado['mensaje']}")

✗ Registro duplicado rechazado correctamente
  → El correo ya está registrado.
